<a href="https://colab.research.google.com/github/polmazon/tfm/blob/claude%2Ffestive-albattani-93veeb/scraper_competencia_honda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Motor de Web Scraping — Ofertas Financieras Competencia Automoción
**Honda Financial Services | TFM BSM Barcelona**

Extrae ofertas de financiación (TIN, TAE, comisión de apertura, cuota, plazo, etc.) de webs de competidores usando scraping + LLM (Claude).

## 1. Instalación de dependencias (ejecutar solo en Colab)

In [ ]:
# Ejecuta esta celda la primera vez en Google Colab (tarda ~1 minuto)
!pip install -q requests beautifulsoup4 "selenium>=4.15" pandas openai
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y -q --fix-missing ./google-chrome-stable_current_amd64.deb
print("Dependencias instaladas correctamente")

## 2. Imports y configuración

In [217]:
import requests
import time
import json
import re
import pandas as pd
from datetime import date
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from openai import OpenAI

print("Librerías cargadas correctamente")

Librerías cargadas correctamente


In [ ]:
from google.colab import userdata

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

CAMPOS_OFERTA = [
    "marca",
    "modelo",
    "precio_vehiculo",
    "cuota_mensual",
    "plazo_meses",
    "entrada",
    "tin",
    "tae",
    "comision_apertura",
    "valor_residual",
    "importe_financiado",
    "tipo_financiacion",
    "banco_financiacion",
    "fecha_fin_oferta",
    "url",
    "fecha_extraccion"
]

print(f"API Key OpenAI cargada: {'OK' if OPENAI_API_KEY else 'ERROR — revisa los Secrets'}")

## 3. Funciones de scraping

In [ ]:
def scrape_estatico(url, reintentos=3, pausa=2):
    for intento in range(reintentos):
        try:
            response = requests.get(url, headers=HEADERS, timeout=15)
            response.raise_for_status()
            return response.text
        except requests.RequestException as e:
            print(f"  [intento {intento+1}/{reintentos}] Error en {url}: {e}")
            time.sleep(pausa * (intento + 1))
    return None


def crear_driver():
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")
    options.add_argument(f"user-agent={HEADERS['User-Agent']}")
    # Selenium Manager descarga automáticamente el chromedriver compatible con el Chrome instalado
    return webdriver.Chrome(options=options)


def scroll_hasta_el_final(driver, pausas=8):
    for i in range(pausas):
        driver.execute_script("window.scrollBy(0, document.body.scrollHeight);")
        time.sleep(1.5)
    driver.execute_script("window.scrollTo(0, 0);")


def scrape_dinamico(url, espera_extra=3, scroll=False):
    driver = crear_driver()
    try:
        driver.get(url)
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )
        time.sleep(espera_extra)
        if scroll:
            scroll_hasta_el_final(driver)
            time.sleep(2)
        return driver.page_source
    except Exception as e:
        print(f"  Error Selenium en {url}: {e}")
        return None
    finally:
        driver.quit()


def html_a_texto(html, seccion_especial=None, max_chars_legal=4000,
                 pagina_listado=False, umbral_tin=6000, tomar_final=False,
                 modelo_hint=None):
    if not html:
        return ""
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "nav", "header", "noscript"]):
        tag.decompose()
    texto = soup.get_text(separator=" ", strip=True)
    texto = re.sub(r'\s+', ' ', texto)

    cabecera = texto[:2500]

    if seccion_especial:
        candidatos = seccion_especial if isinstance(seccion_especial, list) else [seccion_especial]
        for candidato in candidatos:
            posiciones = []
            inicio_busq = 0
            while True:
                p = texto.find(candidato, inicio_busq)
                if p == -1:
                    break
                posiciones.append(p)
                inicio_busq = p + 1
            if not posiciones:
                continue
            pos_elegida = posiciones[0]
            if modelo_hint and len(posiciones) > 1:
                hint_lower = modelo_hint.lower()
                hint_words = [w for w in hint_lower.split() if len(w) > 3]
                mejor = -1
                mejor_pos = posiciones[0]
                for p in posiciones:
                    contexto = texto[p:p + 300].lower()
                    coincidencias = sum(1 for w in hint_words if w in contexto)
                    if coincidencias > mejor:
                        mejor = coincidencias
                        mejor_pos = p
                pos_elegida = mejor_pos
            bloque = texto[pos_elegida:pos_elegida + max_chars_legal]
            n_bloques = len(posiciones)
            print(f"  Sección '{candidato[:40]}' encontrada ({n_bloques} bloque/s, usando pos {pos_elegida})")
            return cabecera + " [...] " + bloque
        print(f"  AVISO: Ninguna sección especial encontrada {[c[:30] for c in candidatos]}, usando fallback")

    if tomar_final:
        bloque = texto[-max_chars_legal:]
        print(f"  [FINAL] Extrayendo últimos {len(bloque)} chars de {len(texto)} totales")
        return cabecera + " [...] " + bloque

    pos = -1
    for kw in ["TIN:", "TIN :", "T.I.N", "TIN ", "TIN%", " TIN "]:
        p = texto.find(kw)
        if p != -1:
            pos = p
            break

    if pos != -1:
        if pagina_listado:
            bloque = texto[max(0, pos - 1500):]
            print(f"  [LISTADO] TIN en pos {pos} — extrayendo {len(bloque)} chars hasta el final")
            return cabecera + " [...] " + bloque
        elif pos < umbral_tin:
            inicio = max(0, pos - 1500)
            bloque = texto[inicio:inicio + max_chars_legal]
            print(f"  Texto enviado al LLM: {len(cabecera) + len(bloque)} chars (TIN en pos {pos})")
            return cabecera + " [...] " + bloque
        else:
            print(f"  TIN encontrado en pos {pos} (carrusel, ignorado) — usando cabecera")

    print(f"  Texto enviado al LLM: {len(cabecera)} chars (sin texto legal propio)")
    return cabecera


print("Funciones de scraping definidas")

## 4. Extracción con LLM (Claude)

In [ ]:
client = OpenAI(api_key=OPENAI_API_KEY)

PROMPT_SISTEMA = """Eres un experto en análisis de ofertas de financiación de automóviles en España.
Tu tarea es extraer información estructurada de textos de páginas web de concesionarios.
Devuelve SIEMPRE un JSON válido con los campos indicados.
Si un campo no aparece en el texto, devuelve null para ese campo.
No inventes datos. Solo extrae lo que esté explícitamente en el texto."""

PROMPT_CAMPOS = """
Para cada oferta devuelve un objeto JSON con estos campos:
- modelo: nombre comercial completo del modelo (incluyendo versión y kW si aparecen)
- tipo_combustible: "gasolina", "diésel", "híbrido", "híbrido enchufable", "eléctrico" o null
- precio_vehiculo: PVP del vehículo en € (número). Busca en este orden:
    1. "PVP al contado" o "Precio al contado" — usa ese valor
    2. Si no hay precio al contado separado, busca "PVP recomendado financiando" o "PVP recomendado" — usa ese valor
    NUNCA inventes ni uses el importe financiado ni la entrada como precio_vehiculo
- precio_financiar: precio por financiar en € (número). Reglas:
    - Si hay DOS precios distintos (uno al contado y otro inferior por financiar), usa el precio inferior
    - Si solo hay UN único precio en el texto, usa ese mismo precio (igual que precio_vehiculo)
    NUNCA dejes este campo null si hay al menos un precio en el texto
- cuota_mensual: cuota mensual TOTAL en € (número). Si la cuota se desglosa en componentes (p.ej. cuota financiera + seguro), usa la suma total
- plazo_meses: duración total del contrato en meses (número)
- entrada: entrada inicial en € (número, 0 si no hay)
- tin: TIN en % (número). Busca "TIN:" o "Tipo Deudor" seguido de un porcentaje
- tae: TAE en % (número). Busca "TAE:" o "T.A.E." seguido de un porcentaje
- comision_apertura: importe de la comisión de apertura en € (número). Busca "Comisión de apertura" seguido de un importe en €. Pon 0 SOLO si el texto dice explícitamente que es gratuita o "sin comisión"
- porcentaje_comision_apertura: comisión de apertura en % sobre el capital (número o null)
- valor_residual: última cuota o valor residual en € (número o null). Busca "Última cuota", "valor residual" o "cuota final"
- importe_financiado: capital total financiado en € (número). Busca "Capital financiado", "Importe financiado" o "Importe total del Crédito"
- tipo_financiacion: nombre exacto del producto financiero
- banco_financiacion: entidad bancaria que OFRECE el préstamo de financiación del vehículo.
    Busca frases como "Financiación ofrecida por", "sujeta a aprobación por parte de" DENTRO del bloque de condiciones financieras (donde aparecen TIN/TAE/cuotas).
    IGNORA completamente cualquier mención a entidades bancarias que aparezca en notas de mantenimiento, garantías u otros servicios posventa: esas menciones NO son el banco financiador del vehículo.
    Ejemplos válidos: "Santander Consumer Finance", "PSA Finance", "Volkswagen Financial Services", "Open Bank, S.A.". null si no aparece.
- fecha_fin_oferta: fecha límite en YYYY-MM-DD (string o null)

Devuelve SOLO este JSON:
{"ofertas": [ {...} ]}
"""


def _llamar_llm(trozo, marca, url, instruccion, max_tokens):
    prompt = f"""{instruccion}

Analiza el siguiente texto de la web de {marca} ({url}).
{PROMPT_CAMPOS}
TEXTO:
{trozo}"""
    respuesta = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        max_tokens=max_tokens,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": PROMPT_SISTEMA},
            {"role": "user", "content": prompt}
        ]
    )
    datos = json.loads(respuesta.choices[0].message.content)
    return [o for o in datos.get("ofertas", []) if o is not None]


def _anotar(ofertas, marca, url):
    for o in ofertas:
        o["marca"] = marca
        o["url"] = url
        o["fecha_extraccion"] = str(date.today())
    return ofertas


_MARCAS_PREFIJOS = {"volkswagen", "toyota", "peugeot", "renault", "nissan", "hyundai", "audi", "seat", "skoda", "honda", "mazda"}

def _fusionar_sin_duplicados(listas):
    """Fusiona listas deduplicando por modelo sin prefijo de marca."""
    vistos, resultado = set(), []
    for o in (o for lista in listas for o in lista):
        nombre = re.sub(r'\s+', ' ', (o.get("modelo") or "").strip().lower())
        palabras = nombre.split()
        if palabras and palabras[0] in _MARCAS_PREFIJOS:
            palabras = palabras[1:]
        key = " ".join(palabras)
        if key and key not in vistos:
            vistos.add(key)
            resultado.append(o)
    return resultado


def extraer_oferta_con_llm(texto, marca, url, filtro_producto=None, pagina_listado=False):
    slug = url.rstrip("/").split("/")[-1]
    pista_modelo = (slug.replace("-easy-plus", "").replace("-easy-renting", "")
                    .replace("-easy", "").replace("-", " ").title())

    try:
        if filtro_producto:
            instruccion = (
                f"IMPORTANTE: El texto puede contener varios bloques legales, uno por versión del modelo. "
                f"Localiza el bloque que corresponde a '{pista_modelo}' "
                f"(busca el bloque cuyo encabezado menciona '{pista_modelo}' o términos similares). "
                f"Extrae ÚNICAMENTE los datos de ESE bloque: PVP al contado (precio_vehiculo), "
                f"precio por financiar (precio_financiar), TIN, TAE, cuota mensual, entrada, "
                f"comisión de apertura, valor residual e importe financiado. "
                f"Devuelve exactamente UNA oferta. No mezcles datos de bloques distintos."
            )
            return _anotar(_llamar_llm(texto, marca, url, instruccion, 3000), marca, url)

        if pagina_listado:
            instruccion = (
                "Extrae TODAS las ofertas de financiación que encuentres en el texto legal. "
                "Cada bloque de condiciones legales corresponde a un modelo distinto. "
                "Devuelve una entrada por cada bloque/modelo con TIN o TAE propio."
            )
            n = len(texto)
            t1, t2 = n // 3, 2 * n // 3
            solape = 6000  # solape amplio para no partir bloques largos
            trozos = [
                texto[:t1 + solape],
                texto[t1 - solape:t2 + solape],
                texto[t2 - solape:]
            ]
            print(f"  [LISTADO] 3 llamadas: {len(trozos[0])} + {len(trozos[1])} + {len(trozos[2])} chars")
            resultados = []
            for i, trozo in enumerate(trozos):
                ofertas_i = _llamar_llm(trozo, marca, url, instruccion, 5000)
                print(f"  Llamada {i+1}/3: {len(ofertas_i)} ofertas")
                resultados.append(_anotar(ofertas_i, marca, url))
                if i < 2:
                    time.sleep(1)
            return _fusionar_sin_duplicados(resultados)

        instruccion = "Extrae la oferta de financiación principal que encuentres."
        return _anotar(_llamar_llm(texto, marca, url, instruccion, 3000), marca, url)

    except Exception as e:
        print(f"  Error LLM para {url}: {e}")
        return []


print("Cliente OpenAI (gpt-4o-mini, temperature=0) configurado")

## 5. Pipeline completo: scraping + extracción LLM

In [ ]:
def procesar_url(url, marca, usar_selenium=False, scroll=False,
                 seccion_especial=None, filtro_producto=None,
                 pagina_listado=False, max_chars_legal=4000, umbral_tin=6000,
                 tomar_final=False):
    print(f"Procesando: {marca} — {url}")
    html = scrape_dinamico(url, scroll=scroll) if usar_selenium else scrape_estatico(url)
    if not html:
        print(f"  No se pudo descargar {url}")
        return []
    slug = url.rstrip("/").split("/")[-1]
    pista_modelo_hint = (slug.replace("-easy-plus", "").replace("-easy-renting", "")
                         .replace("-easy", "").replace("-", " "))
    texto = html_a_texto(
        html,
        seccion_especial=seccion_especial,
        max_chars_legal=max_chars_legal,
        pagina_listado=pagina_listado,
        umbral_tin=umbral_tin,
        tomar_final=tomar_final,
        modelo_hint=pista_modelo_hint if filtro_producto else None
    )
    if not pagina_listado and not tomar_final and not any(kw in texto for kw in ["TIN:", "TIN :", "T.I.N", "TIN%", " TIN "]):
        print(f"  Sin TIN en el texto — página sin oferta financiera, saltando")
        return []
    ofertas = extraer_oferta_con_llm(
        texto, marca, url,
        filtro_producto=filtro_producto,
        pagina_listado=pagina_listado
    )
    print(f"  Ofertas encontradas: {len(ofertas)}")
    return ofertas


def descubrir_urls_toyota():
    """Extrae desde toyota.es/promociones todas las URLs con 'easy' sin 'renting'."""
    BASE = "https://www.toyota.es"
    print(f"Descubriendo URLs Toyota Easy desde {BASE}/promociones ...")
    html = scrape_estatico(f"{BASE}/promociones")
    if not html:
        print("  No se pudo descargar la página de promociones de Toyota")
        return []
    soup = BeautifulSoup(html, "html.parser")
    urls = []
    EXCLUIR = ["/promociones/toyota-easy-plus", "/promociones/toyota-easy",
               "/promociones/toyota-easy-complet"]
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if ("/promociones/" in href
                and "easy" in href
                and "renting" not in href
                and not any(href.endswith(ex.split("/")[-1]) and "/finance-insurance/" not in href
                            for ex in EXCLUIR)
                and "/finance-insurance/" not in href):
            url_completa = href if href.startswith("http") else BASE + href
            if url_completa not in urls:
                urls.append(url_completa)
    print(f"  URLs Easy encontradas: {len(urls)}")
    for u in urls:
        print(f"    {u}")
    return urls


def descubrir_urls_renault():
    """Extrae desde promociones.renault.es/particulares/ solo URLs de modelos de coche."""
    BASE = "https://promociones.renault.es"
    INDEX = BASE + "/particulares/"
    print(f"Descubriendo URLs Renault desde {INDEX} ...")
    html = scrape_dinamico(INDEX, scroll=False)
    if not html:
        print("  No se pudo descargar la página de Renault")
        return []
    soup = BeautifulSoup(html, "html.parser")
    urls = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        partes = href.rstrip("/").split("/")
        if not ("/particulares/" in href
                and len([p for p in partes if p]) >= 2
                and not href.rstrip("/").endswith("/particulares")):
            continue
        # Excluir URLs con dígitos en el slug (son promociones genéricas, no modelos)
        slug = partes[-1] if partes[-1] else partes[-2]
        if any(c.isdigit() for c in slug):
            continue
        url_completa = href if href.startswith("http") else BASE + href
        if url_completa not in urls:
            urls.append(url_completa)
    print(f"  URLs Renault encontradas: {len(urls)}")
    for u in urls:
        print(f"    {u}")
    return urls


def descubrir_urls_hyundai():
    """Extrae desde hyundai.com/es/es/modelos.html todas las URLs de modelos."""
    BASE = "https://www.hyundai.com"
    INDEX = BASE + "/es/es/modelos.html"
    print(f"Descubriendo URLs Hyundai desde {INDEX} ...")
    html = scrape_dinamico(INDEX, scroll=True)
    if not html:
        print("  No se pudo descargar la página de Hyundai")
        return []
    soup = BeautifulSoup(html, "html.parser")
    urls = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if "/es/es/modelos/" in href and href.endswith(".html"):
            segmento = href.rstrip("/").split("/es/es/modelos/")[-1]
            if ("/" in segmento or "configurador" in href
                    or "coches" in href or href.endswith("/modelos.html")):
                continue
            url_completa = href if href.startswith("http") else BASE + href
            if url_completa not in urls:
                urls.append(url_completa)
    print(f"  URLs Hyundai encontradas: {len(urls)}")
    for u in urls:
        print(f"    {u}")
    return urls


def descubrir_urls_mazda():
    """Extrae desde mazda.es/promociones/promociones-actuales/ las URLs de modelos con oferta."""
    BASE = "https://www.mazda.es"
    INDEX = BASE + "/promociones/promociones-actuales/"
    print(f"Descubriendo URLs Mazda desde {INDEX} ...")
    html = scrape_dinamico(INDEX, scroll=True)
    if not html:
        print("  No se pudo descargar la página de Mazda")
        return []
    soup = BeautifulSoup(html, "html.parser")
    urls = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        # Solo aceptar URLs bajo /promociones-actuales/ (excluye /como-comprar/ y otras secciones)
        if "/promociones-actuales/" not in href:
            continue
        url_completa = href if href.startswith("http") else BASE + href
        if not url_completa.startswith(BASE):
            continue
        slug = url_completa.rstrip("/").split("/")[-1]
        # Excluir la página índice
        if slug in ("promociones-actuales", ""):
            continue
        if url_completa not in urls:
            urls.append(url_completa)
    print(f"  URLs Mazda encontradas: {len(urls)}")
    for u in urls:
        print(f"    {u}")
    return urls


print("Pipeline y auto-descubrimiento Toyota/Renault/Hyundai/Mazda definidos")

## 6. URLs de la competencia

In [ ]:
COMPETENCIA = {
    "TOYOTA": {
        "selenium": True, "scroll": True,
        "seccion_especial": ["Precio correspondiente a", "Precio por financiar:", "Toyota Easy Plus:", "Oferta financiera con el producto Toyota Easy"], "filtro_producto": "Easy Plus",
        "pagina_listado": False, "max_chars_legal": 10000, "tomar_final": False,
        "umbral_tin": 999999,
        "urls": []  # se auto-descubren desde toyota.es/promociones
    },
    "VOLKSWAGEN": {
        "selenium": True, "scroll": True,
        "seccion_especial": None, "filtro_producto": None,
        "pagina_listado": True, "max_chars_legal": 60000,
        "urls": ["https://www.volkswagen.es/es/ofertas.html"]
    },
    "PEUGEOT": {
        "selenium": True, "scroll": True,
        "seccion_especial": None, "filtro_producto": None,
        "pagina_listado": True, "max_chars_legal": 60000,
        "urls": ["https://www.peugeot.es/comprar/ofertas-del-momento.html"]
    },
    "RENAULT": {
        "selenium": True, "scroll": False,
        "seccion_especial": "CONDICIONES LEGALES PARA PENÍNSULA Y BALEARES",
        "filtro_producto": None,
        "pagina_listado": False, "max_chars_legal": 4000,
        "urls": []  # se auto-descubren desde promociones.renault.es/particulares/
    },
    "NISSAN": {
        "selenium": True, "scroll": True,
        "seccion_especial": None, "filtro_producto": None,
        "pagina_listado": True, "max_chars_legal": 60000,
        "urls": ["https://www.nissan.es/vehiculos/ofertas.html"]
    },
    "HYUNDAI": {
        "selenium": True, "scroll": True,
        "seccion_especial": None, "filtro_producto": None,
        "pagina_listado": False, "max_chars_legal": 8000,
        "umbral_tin": 20000, "tomar_final": True,
        "urls": []  # se auto-descubren desde hyundai.com/es/es/modelos.html
    },
    "AUDI": {
        "selenium": True, "scroll": True,
        "seccion_especial": None, "filtro_producto": None,
        "pagina_listado": True, "max_chars_legal": 60000,
        "urls": ["https://www.audi.es/es/compra/promociones/"]
    },
    "HONDA": {
        "selenium": True, "scroll": True,
        "seccion_especial": None, "filtro_producto": None,
        "pagina_listado": True, "max_chars_legal": 60000,
        "urls": ["https://www.honda.es/cars/offers.html"]
    },
    "MAZDA": {
        "selenium": True, "scroll": True,
        "seccion_especial": None, "filtro_producto": None,
        "pagina_listado": False, "max_chars_legal": 8000,
        "umbral_tin": 999999, "tomar_final": True,
        "urls": []  # se auto-descubren desde mazda.es/promociones/promociones-actuales/
    }
}

print(f"Configuradas {len(COMPETENCIA)} marcas")

## 7. Ejecución del scraping

In [ ]:
MARCAS_A_EJECUTAR = ["TOYOTA", "VOLKSWAGEN", "PEUGEOT", "RENAULT", "NISSAN", "HYUNDAI", "AUDI", "MAZDA", "HONDA"]
# Para probar una sola marca: MARCAS_A_EJECUTAR = ["MAZDA"]

todas_las_ofertas = []

for marca in MARCAS_A_EJECUTAR:
    config = COMPETENCIA[marca]
    print(f"\n{'='*50}\nMARCA: {marca}\n{'='*50}")

    # Auto-descubrimiento de URLs por marca
    if marca == "TOYOTA":
        urls = descubrir_urls_toyota()
        if not urls:
            print("  Fallback a URLs hardcodeadas")
            urls = config["urls"]
    elif marca == "RENAULT":
        urls = descubrir_urls_renault()
        if not urls:
            print("  Fallback a URLs hardcodeadas")
            urls = config["urls"]
    elif marca == "HYUNDAI":
        urls = descubrir_urls_hyundai()
        if not urls:
            print("  Fallback a URLs hardcodeadas")
            urls = config["urls"]
    elif marca == "MAZDA":
        urls = descubrir_urls_mazda()
        if not urls:
            print("  Fallback a URLs hardcodeadas")
            urls = config["urls"]
    else:
        urls = config["urls"]

    if not urls:
        print(f"  Sin URLs para {marca}, saltando.")
        continue

    for url in urls:
        ofertas = procesar_url(
            url, marca,
            usar_selenium=config["selenium"],
            scroll=config.get("scroll", False),
            seccion_especial=config.get("seccion_especial"),
            filtro_producto=config.get("filtro_producto"),
            pagina_listado=config.get("pagina_listado", False),
            max_chars_legal=config.get("max_chars_legal", 4000),
            umbral_tin=config.get("umbral_tin", 6000),
            tomar_final=config.get("tomar_final", False)
        )
        todas_las_ofertas.extend(ofertas)
        time.sleep(2)

print(f"\n{'='*50}")
print(f"RESUMEN: {len(todas_las_ofertas)} ofertas brutas extraídas")
marcas_con_datos = set(o["marca"] for o in todas_las_ofertas)
print(f"Marcas CON datos: {marcas_con_datos}")
marcas_sin_datos = set(MARCAS_A_EJECUTAR) - marcas_con_datos
if marcas_sin_datos:
    print(f"Marcas SIN datos: {marcas_sin_datos}")

In [ ]:
df_bruto = pd.DataFrame(todas_las_ofertas)

if df_bruto.empty:
    print("No se han extraído ofertas.")
else:
    df = (
        df_bruto
        .drop_duplicates(subset=["marca", "modelo"], keep="first")
        .reset_index(drop=True)
    )

    # Eliminar filas sin datos financieros reales
    cols_financieras = ["tin", "tae", "cuota_mensual"]
    mask_con_datos = df[cols_financieras].notna().any(axis=1)
    descartadas = (~mask_con_datos).sum()
    if descartadas > 0:
        print(f"  Descartando {descartadas} filas sin datos financieros (TIN/TAE/cuota todos nulos)")
    df = df[mask_con_datos].reset_index(drop=True)

    # Clasificar tipo de financiación: PCP si hay valor residual > 0, HP si no
    df["tipo_financiacion"] = df.apply(
        lambda r: "PCP" if (pd.notna(r.get("valor_residual")) and float(r.get("valor_residual") or 0) > 0)
                  else "HP",
        axis=1
    )

    # Corregir precios: si precio_financiar > precio_vehiculo el LLM cogió un precio "desde"
    # del carrusel como PVP. En ese caso precio_vehiculo = precio_financiar y promocion = 0.
    if "precio_vehiculo" in df.columns and "precio_financiar" in df.columns:
        pv = pd.to_numeric(df["precio_vehiculo"], errors="coerce")
        pf = pd.to_numeric(df["precio_financiar"], errors="coerce")
        mask_invertido = pf > pv
        if mask_invertido.any():
            print(f"  Corrigiendo {mask_invertido.sum()} filas con precio_financiar > precio_vehiculo")
            df.loc[mask_invertido, "precio_vehiculo"] = pf[mask_invertido]
            pv = pd.to_numeric(df["precio_vehiculo"], errors="coerce")
            pf = pd.to_numeric(df["precio_financiar"], errors="coerce")
        df["promocion_financiacion"] = (pv - pf).round(2).fillna(0)

    # Estandarizar plazo: 48→49, 36→37 (período comercial normalizado)
    if "plazo_meses" in df.columns:
        df["plazo_meses"] = pd.to_numeric(df["plazo_meses"], errors="coerce")
        df["plazo_meses"] = df["plazo_meses"].replace({48: 49, 36: 37})

    CAMPOS_ORDENADOS = [
        "marca", "modelo", "tipo_combustible", "precio_vehiculo", "precio_financiar",
        "promocion_financiacion", "cuota_mensual", "plazo_meses", "entrada", "tin", "tae",
        "porcentaje_comision_apertura", "comision_apertura",
        "valor_residual", "importe_financiado", "tipo_financiacion",
        "fecha_fin_oferta", "banco_financiacion", "url", "fecha_extraccion"
    ]
    cols = [c for c in CAMPOS_ORDENADOS if c in df.columns]
    df = df[cols]

    pd.set_option("display.max_columns", None)
    pd.set_option("display.max_rows", 100)

    print(f"Ofertas brutas: {len(df_bruto)} → tras deduplicar y filtrar: {len(df)}")
    print(f"\nModelos por marca:")
    print(df.groupby("marca")["modelo"].count().to_string())
    display(df)

In [224]:
# Exportar a CSV
nombre_archivo = f"ofertas_competencia_{date.today().strftime('%Y%m%d')}.csv"
df.to_csv(nombre_archivo, index=False, encoding="utf-8-sig")
print(f"Guardado en: {nombre_archivo}")

# En Colab, descargar el archivo:
# from google.colab import files
# files.download(nombre_archivo)

Guardado en: ofertas_competencia_20260614.csv


In [225]:
# Resumen comparativo por marca
if not df.empty and "marca" in df.columns:
    resumen = df.groupby("marca").agg(
        num_ofertas=("modelo", "count"),
        tin_medio=("tin", "mean"),
        tae_medio=("tae", "mean"),
        cuota_min=("cuota_mensual", "min"),
        cuota_max=("cuota_mensual", "max"),
        mean_com_apertura=("porcentaje_comision_apertura", "mean")
    ).round(2)
    print(resumen)

            num_ofertas  tin_medio  tae_medio  cuota_min  cuota_max
marca                                                              
TOYOTA               18       7.03       8.24       99.0      495.0
VOLKSWAGEN           21       6.95       8.71      100.0      350.0


## 8. Exportar resultados al Excel de análisis de precios

Esta celda lee el CSV generado por el scraper y actualiza la plantilla Excel con los datos de la competencia, pestaña por pestaña. **No modifica ninguna celda anterior.** Sube el archivo `02._CAR_JUNE_2026__PRICE_COMPETENCE_ANALISIS.xlsx` a Colab antes de ejecutar.


In [ ]:
# ============================================================
# CELDA 8 — Exportar datos al Excel de análisis de precios
# - Columnas Honda (azul): datos scrapeados de honda.es
# - Columnas competencia (amarillo): datos de las demás marcas
# - Rojo: competidor/modelo sin coincidencia o sin dato
# ============================================================
!pip install -q openpyxl rapidfuzz

import os, glob
from openpyxl import load_workbook
from openpyxl.styles import PatternFill
import pandas as pd
from rapidfuzz import process, fuzz

FILL_HONDA    = PatternFill(start_color="ADD8E6", end_color="ADD8E6", fill_type="solid")  # azul claro
FILL_UPDATED  = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")  # amarillo
FILL_NO_MATCH = PatternFill(start_color="FF4444", end_color="FF4444", fill_type="solid")  # rojo

# --- 1. CSV ---
csvs = sorted(glob.glob("ofertas_competencia_*.csv"), reverse=True)
if csvs:
    csv_path = csvs[0]
    print(f"CSV encontrado automáticamente: {csv_path}")
else:
    print("Selecciona el CSV desde tu ordenador:")
    from google.colab import files
    csv_path = list(files.upload().keys())[0]

df_csv = pd.read_csv(csv_path)
print(f"CSV cargado: {len(df_csv)} filas")

# --- 2. Excel plantilla ---
xl_found = next(
    (f for f in sorted(glob.glob("*.xlsx"))
     if any(k in f.lower() for k in ["price", "car", "analisis"])),
    None
)
if xl_found:
    EXCEL_TEMPLATE = xl_found
    print(f"Excel encontrado: {EXCEL_TEMPLATE}")
else:
    print("Selecciona el Excel plantilla desde tu ordenador:")
    from google.colab import files
    EXCEL_TEMPLATE = list(files.upload().keys())[0]

wb = load_workbook(EXCEL_TEMPLATE)
print(f"Pestañas disponibles: {wb.sheetnames}")

# --- 3. Detección de tipo de motorización ---
TIPOS_MOTOR = [
    ("electrico", ["bev","eléctric","electric","e-2008","e-208","e-308","e-3008",
                   "e-408","e-5008","ioniq","zoe","megane e","ariya","leaf",
                   "micra","inster","mazda6e","kona eléctrico"]),
    ("phev",      ["phev","plug-in","plug in","recargable","e:phev",
                   "rav4 plug","tucson phev","santa fe phev"]),
    ("hibrido",   ["hybrid","híbrido","hibrido","hev","self-charging","i-mmd",
                   "mild hybrid","mhev","full hybrid","e-tech","e-power",
                   "e-skyactiv","48v","yaris","corolla","rav4","jazz","hr-v",
                   "crv","cr-v","crosstar","zr-v","civic","prelude",
                   "captur","symbioz","austral","juke","qashqai",
                   "kona híbrido","tucson híbrido","santa fe híbrido"]),
    ("gasolina",  ["tsi","gdi","1.0t","1.5t","tfsi","gasolina","petrol","turbo",
                   "tce","puretech","dig-t","sce","dci","tdi","diesel","diésel",
                   "mpi","eco-g","glp"]),
]

def detectar_tipo(texto):
    if not texto: return None
    t = str(texto).lower()
    for tipo, kws in TIPOS_MOTOR:
        if any(k in t for k in kws): return tipo
    return None

df_csv["_tipo"] = df_csv["modelo"].apply(detectar_tipo)
df_honda = df_csv[df_csv["marca"].str.upper() == "HONDA"].copy()

# --- 4. Búsqueda fuzzy genérica ---
def buscar(df, nombre, plazo_ref=None):
    tipo = detectar_tipo(nombre)
    sub = df[df["_tipo"] == tipo] if tipo else df
    if sub.empty: sub = df
    res = process.extractOne(
        nombre, sub["modelo"].fillna("").tolist(),
        scorer=fuzz.token_set_ratio, score_cutoff=55
    )
    aviso = ""
    if res is None and tipo:
        res = process.extractOne(
            nombre, df["modelo"].fillna("").tolist(),
            scorer=fuzz.token_set_ratio, score_cutoff=55
        )
        if res is None: return None, 0, None, tipo, ""
        aviso = " ⚠️ (distinto tipo motor)"
        sub = df
    elif res is None:
        return None, 0, None, tipo, ""
    sub2 = sub[sub["modelo"].fillna("") == res[0]].copy()
    if plazo_ref and "plazo_meses" in sub2.columns:
        pp = sub2[sub2["plazo_meses"] == plazo_ref]
        if not pp.empty: sub2 = pp
    if "cuota_mensual" in sub2.columns:
        sub2 = sub2.sort_values("cuota_mensual")
    return sub2.iloc[0], res[1], res[0], tipo, aviso

def norm_pct(v):
    try: f = float(v); return f / 100 if f > 1 else f
    except: return v

# --- 5. Campos a actualizar: fila Excel → (columna CSV, es_porcentaje) ---
CAMPOS = {
    5:  ("cuota_mensual",               False),
    6:  ("entrada",                     False),
    7:  ("tin",                         True),
    8:  ("porcentaje_comision_apertura", True),
    9:  ("tae",                         True),
    10: ("plazo_meses",                 False),
    11: ("valor_residual",              False),
    13: ("precio_financiar",            False),
}

def escribir_campos(ws, col_idx, oferta, fill):
    for fila, (campo, es_pct) in CAMPOS.items():
        c = ws.cell(row=fila, column=col_idx)
        if campo in oferta.index and pd.notna(oferta[campo]):
            c.value = norm_pct(oferta[campo]) if es_pct else oferta[campo]
            c.fill = fill
        else:
            c.value = None
            c.fill = FILL_NO_MATCH

# --- 6. Configuración: columnas Honda por pestaña ---
# col = índice openpyxl (D=4, I=9), modelo = texto para fuzzy match
HONDA_COLS = {
    "MONTHLY JAZZ":     [{"col": 4, "modelo": "Jazz"},
                         {"col": 9, "modelo": "Jazz Crosstar"}],
    "MONTHLY HRV":      [{"col": 4, "modelo": "HR-V"}],
    "MONTHLY CIVIC":    [{"col": 4, "modelo": "Civic"}],
    "MONTHLY ZRV":      [{"col": 4, "modelo": "ZR-V"}],
    "MONTHLY CRV FHEV": [{"col": 4, "modelo": "CR-V"}],
    "MONTHLY CRV PHEV": [{"col": 4, "modelo": "CR-V e:PHEV"}],
}

TABS_PLAZO = {
    "MONTHLY JAZZ":     36,
    "MONTHLY HRV":      36,
    "MONTHLY CIVIC":    36,
    "MONTHLY ZRV":      36,
    "MONTHLY CRV FHEV": 36,
    "MONTHLY CRV PHEV": 36,
}
sheet_map = {s.upper(): s for s in wb.sheetnames}

# --- 7. Rellenar cada pestaña ---
for tab_key, plazo_ref in TABS_PLAZO.items():
    real = sheet_map.get(tab_key)
    if not real:
        print(f"⚠️  '{tab_key}' no encontrada")
        continue
    ws = wb[real]
    print(f"\n📋 {real}")

    # 7a. Columnas Honda (azul)
    if not df_honda.empty:
        for h in HONDA_COLS.get(tab_key, []):
            oferta, score, match, tipo, aviso = buscar(df_honda, h["modelo"], plazo_ref)
            if oferta is not None:
                escribir_campos(ws, h["col"], oferta, FILL_HONDA)
                print(f"  🔵 Honda col {h['col']} '{h['modelo']}' → '{match}' ({score}%)")
            else:
                for fila in CAMPOS:
                    c = ws.cell(row=fila, column=h["col"])
                    c.value = None; c.fill = FILL_NO_MATCH
                print(f"  ❌ Honda col {h['col']} '{h['modelo']}' — sin datos en CSV")
    else:
        print("  ℹ️  Sin filas Honda en el CSV (ejecuta el scraper con HONDA incluido)")

    # 7b. Columnas competencia (amarillo / rojo)
    vacias = 0
    actualizados = 0
    for col_idx in range(5, 31):
        cn = ws.cell(row=4, column=col_idx)
        nombre = cn.value
        if not nombre:
            vacias += 1
            if vacias >= 3: break
            continue
        vacias = 0

        oferta, score, match, tipo, aviso = buscar(df_csv, str(nombre), plazo_ref)
        tipo_tag = f"[{tipo}]" if tipo else "[?]"

        if oferta is not None:
            escribir_campos(ws, col_idx, oferta, FILL_UPDATED)
            actualizados += 1
            print(f"  ✅ {tipo_tag} '{nombre}' → '{match}' ({score}%){aviso}")
        else:
            cn.fill = FILL_NO_MATCH
            for fila in CAMPOS:
                c = ws.cell(row=fila, column=col_idx)
                c.value = None; c.fill = FILL_NO_MATCH
            print(f"  ❌ {tipo_tag} '{nombre}' — sin coincidencia")

    print(f"  → {actualizados} competidores actualizados")

# --- 8. Guardar y descargar ---
OUTPUT = "02._CAR_PRICE_COMPETENCE_ACTUALIZADO.xlsx"
wb.save(OUTPUT)
print(f"\n✅ Excel guardado: {OUTPUT}")
try:
    from google.colab import files
    files.download(OUTPUT)
    print("📥 Descarga iniciada automáticamente.")
except Exception:
    print(f"Descarga manual: panel de archivos de Colab → '{OUTPUT}'.")
